In [1]:
import os
import re
import mlflow
from tqdm import tqdm
import unstructured
from unstructured.partition.pdf import partition_pdf


c:\Users\HP\RAG_AI\Rag_Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("RAG_Data_Ingestion")

<Experiment: artifact_location='file:///c:/Users/HP/RAG_AI/Rag_Project/mlruns/1', creation_time=1773118779468, experiment_id='1', last_update_time=1773118779468, lifecycle_stage='active', name='RAG_Data_Ingestion', tags={}, workspace='default'>

In [4]:
DATA_FOLDER = r"C:\Users\HP\RAG_AI\Rag_Project\data"



In [5]:
# -------------------------------
# Text Cleaning Function
# -------------------------------
def clean_text(text):

    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"Page \d+", "", text)
    text = text.strip()

    return text

In [6]:
# -------------------------------
# PDF Processing
# -------------------------------
def process_pdf(pdf_path):

    elements = partition_pdf(
        filename=pdf_path,
        extract_images_in_pdf=True,
        infer_table_structure=True,
        strategy="fast"   # ensures no OCR
    )

    cleaned_chunks = []

    for el in elements:

        if hasattr(el, "text") and el.text:
            text = clean_text(el.text)

            if len(text) > 50:
                cleaned_chunks.append(text)

    return cleaned_chunks


In [7]:
# -------------------------------
# Ingestion Pipeline
# -------------------------------
def run_ingestion():

    mlflow.set_experiment("RAG_Data_Ingestion")

    with mlflow.start_run():

        pdf_files = [
            os.path.join(DATA_FOLDER, f)
            for f in os.listdir(DATA_FOLDER)
            if f.endswith(".pdf")
        ]

        all_chunks = []

        for i, pdf in enumerate(pdf_files):

            chunks = process_pdf(pdf)

            all_chunks.extend(chunks)

            mlflow.log_metric(f"chunks_file_{i}", len(chunks))
            
        print(f"\nTotal Chunks Extracted: {len(all_chunks)}")

        mlflow.log_metric("total_pdfs", len(pdf_files))
        mlflow.log_metric("total_chunks", len(all_chunks))

        return all_chunks

In [8]:
if __name__ == "__main__":

    chunks = run_ingestion()

    print("\nSample Chunk:\n")
    print(chunks[0])

Cannot set non-stroke color: /'P0' is an invalid float value
Cannot set non-stroke color: /'P0' is an invalid float value
Cannot set non-stroke color: /'P0' is an invalid float value
Cannot set non-stroke color: /'P0' is an invalid float value
Cannot set non-stroke color: /'P0' is an invalid float value



Total Chunks Extracted: 1756
🏃 View run polite-kit-580 at: http://127.0.0.1:5000/#/experiments/1/runs/88b4ad1f9be642e28fb891179ba72d45
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Sample Chunk:

A joint initiative to impart farmers with technical knowledge on basic agriculture.
